In [2]:
pip install groq python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import joblib
import pandas as pd
import json
from dotenv import load_dotenv
from groq import Groq

# 1. Chargement de la clé API depuis .env
load_dotenv('../.env') # Charge le .env situé à la racine
groq_api_key = os.getenv("GROQ_API_KEY")

client = Groq(api_key=groq_api_key)

# 2. Chargement du modèle ML
model_data = joblib.load('../models/mro_risk_model.pkl')
model = model_data['model']
features = model_data['features']

print(" Modèle et Clé Groq chargés avec succès !")

# 3. Fonction d'inférence ML
def predict_late_risk(scenario_dict):
    df_single = pd.DataFrame([scenario_dict])
    df_encoded = pd.get_dummies(df_single)
    df_aligned = df_encoded.reindex(columns=features, fill_value=0)
    
    # Calcul de la probabilité de retard
    risk_proba = model.predict_proba(df_aligned)[0][1]
    return risk_proba

# 4. Agent LLM Groq (MRO Decision Engine)
def run_mro_llm_agent(scenario, risk_score):
    system_prompt = """
    Tu es un Expert Senior en Supply Chain Aéronautique et MRO (Maintenance, Repair, and Overhaul) pour une compagnie aérienne (ex: Royal Air Maroc)[cite: 1].
    Ton rôle est d'analyser le score de risque produit par un modèle ML et de fournir un plan d'action préventif clair et structuré pour éviter une immobilisation d'avion (AOG - Aircraft On Ground).

    Structure ta réponse ainsi :
    1.  **Diagnostic du Risque** (Explication métier du score ML)
    2.  **Impact Opérationnel MRO** (Risque AOG / Maintenance)
    3.  **Plan d'Action Immédiat** (2 à 3 mesures logistiques concrètes)
    """

    user_prompt = f"""
    Alerte Commande Pièce Aéronautique :
    - Données Logistiques : {json.dumps(scenario, indent=2)}
    - Risque de Retard Détecté par le ML : {risk_score * 100:.1f}%

    Rédige ton analyse et tes recommandations métiers.
    """

    # Utilisation du modèle Llama-3 rapide sur Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

 Modèle et Clé Groq chargés avec succès !


In [4]:
# 5. TEST COMPLET DU PIPELINE
sample_order = {
    'ordered_qty': 50,
    'promised_lead_time': 90,  # Délais long (90 jours)
    'qty_fill_rate': 0.70       # Fournisseur qui ne livre pas à 100%
}

# Inférence ML
score = predict_late_risk(sample_order)
print(f"Probabilité de retard (ML) : {score * 100:.2f}%\n")

# Génération par l'Agent LLM
recommendation = run_mro_llm_agent(sample_order, score)
print("=== RECOMMANDATION DE L'AGENT IA (GROQ) ===")
print(recommendation)

Probabilité de retard (ML) : 71.10%

=== RECOMMANDATION DE L'AGENT IA (GROQ) ===
**Analyse et Recommandations**

### 1. **Diagnostic du Risque**

Le score de risque produit par le modèle ML est de 71,1%, ce qui indique un risque élevé de retard dans la livraison de la pièce aéronautique commandée. Ce risque est calculé en fonction des données logistiques fournies, notamment la quantité commandée (50 unités), le délai de livraison promis (90 jours) et le taux de remplissage des commandes (0,7 ou 70%). 

Le modèle ML a probablement pris en compte ces facteurs pour estimer que les fournisseurs pourraient avoir des difficultés à respecter le délai de livraison promis, ce qui pourrait entraîner des retards. Étant donné que le taux de remplissage des commandes est de 70%, cela signifie que seulement 70% des commandes similaires ont été livrées dans les délais prévus par le passé, laissant une marge de 30% pour les retards.

### 2. **Impact Opérationnel MRO**

Un retard dans la livraison de p